# セファイド変光星の周期−光度関係で大マゼラン雲までの距離を求める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/distance/cepheid.ipynb)

**古典的セファイド変光星** は、数日〜数十日の周期で明るさが規則正しく変わる明るい巨星です。
1912 年にヘンリエッタ・リービットが、「周期が長いセファイドほど本来の明るさが明るい」ことを見つけました。これが **周期−光度関係**(リービットの法則)です。

$$
M = a + b \log_{10} P
$$

周期 $P$ は測れば分かるので、この関係から絶対等級 $M$ が分かり、見かけの等級と比べて距離指数が求まります。
セファイドは太陽の数千〜数万倍明るいので、ほかの銀河の中でも見つけられます。

このノートブックでやること:

1. Gaia DR3 で見つかった古典的セファイドを取得する
2. 天の川銀河の中で視差がよく分かっているセファイドで、周期−光度関係を **自分で較正** する
3. その関係を大マゼラン雲(LMC)のセファイドに当てはめ、LMC までの距離を求める

## 減光をどうするか: ヴェーゼンハイト等級

天の川銀河のセファイドの多くは銀河面にあって、星間塵による減光が大きく、しかも星ごとに違います。
そこで、減光の影響が打ち消されるように等級と色を組み合わせた **ヴェーゼンハイト等級** を使います。

$$
W = G - \lambda\,(G_{BP} - G_{RP}), \qquad \lambda \approx 1.90
$$

減光で星が $A_G$ だけ暗くなると色も赤くなりますが、$\lambda$ を減光則に合わせて選んでおくと、2 つの効果がほぼ打ち消し合います
(Gaia バンドでの $\lambda = 1.90$ は Ripepi et al. 2019 などで使われている値)。

## 使うデータ(すべてオープン)

- **Gaia DR3** (ESA): `gaiadr3.vari_cepheid`(セファイドの周期・平均等級・分類)、`gaiadr3.gaia_source`(視差・位置)

## 0. 準備

In [ ]:
# Colab には astroquery が入っていないのでインストールする
!pip install -q astroquery

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord

from astroquery.gaia import Gaia

Gaia.ROW_LIMIT = -1

LAMBDA_W = 1.90  # ヴェーゼンハイト等級の係数


def distance_from_modulus(mu):
    # 距離指数 m - M → 距離 [pc]
    return 10.0 ** ((mu + 5.0) / 5.0)

## 1. Gaia DR3 のセファイドを取得する

`vari_cepheid` には、Gaia の測光から変光星と判定された星の周期や平均等級が入っています。
ここでは

- `type_best_classification = 'DCEP'`: 古典的セファイド(II 型セファイドなどは別の周期−光度関係に従うので除く)
- `mode_best_classification = 'FUNDAMENTAL'`: 基本振動モード(倍音で振動する星は周期がずれるので除く)

に絞ります。等級は、明るさの変化を 1 周期で平均した値 `int_average_*` を使います。

In [ ]:
query = '''
SELECT c.source_id, c.pf, c.int_average_g, c.int_average_bp, c.int_average_rp,
       g.ra, g.dec, g.parallax, g.parallax_error, g.ruwe
FROM gaiadr3.vari_cepheid AS c
JOIN gaiadr3.gaia_source AS g USING (source_id)
WHERE c.type_best_classification = 'DCEP'
  AND c.mode_best_classification = 'FUNDAMENTAL'
  AND c.pf IS NOT NULL
  AND c.int_average_g IS NOT NULL
  AND c.int_average_bp IS NOT NULL
  AND c.int_average_rp IS NOT NULL
'''
job = Gaia.launch_job_async(query)
ceph = job.get_results().to_pandas()
ceph.columns = ceph.columns.str.lower()

ceph["logp"] = np.log10(ceph["pf"])  # pf: 基本振動の周期 [日]
ceph["w"] = ceph["int_average_g"] - LAMBDA_W * (ceph["int_average_bp"] - ceph["int_average_rp"])
print(len(ceph), "件")
ceph.head()

### 天の川銀河 / LMC / SMC に分ける

大マゼラン雲(LMC)と小マゼラン雲(SMC)の方向にある星を、中心からの角距離で切り分けます。

In [ ]:
LMC_CENTER = SkyCoord(80.894 * u.deg, -69.756 * u.deg)
SMC_CENTER = SkyCoord(13.187 * u.deg, -72.829 * u.deg)
LMC_RADIUS, SMC_RADIUS = 8.0, 4.0  # [deg]

coords = SkyCoord(ceph["ra"].to_numpy() * u.deg, ceph["dec"].to_numpy() * u.deg)
ceph["sep_lmc"] = coords.separation(LMC_CENTER).deg
ceph["sep_smc"] = coords.separation(SMC_CENTER).deg

in_lmc = ceph["sep_lmc"] < LMC_RADIUS
in_smc = ceph["sep_smc"] < SMC_RADIUS
lmc = ceph[in_lmc].copy()
mw_all = ceph[~in_lmc & ~in_smc].copy()  # アンドロメダ銀河などの星もわずかに混じるが、下の視差の条件で落ちる
print(f"LMC: {len(lmc)}, SMC: {in_smc.sum()}, それ以外: {len(mw_all)}")

plt.figure(figsize=(10, 5))
plt.scatter(ceph["ra"], ceph["dec"], s=1, c="gray", label="all DCEP (F)")
plt.scatter(lmc["ra"], lmc["dec"], s=1, c="tab:red", label="LMC")
plt.xlabel("RA [deg]")
plt.ylabel("Dec [deg]")
plt.gca().invert_xaxis()
plt.legend(markerscale=5)
plt.show()

## 2. 天の川銀河のセファイドで周期−光度関係を較正する

視差の精度がよい($p/\sigma_p > 10$)セファイドだけを使い、視差から絶対ヴェーゼンハイト等級を出します。

$$
M_W = W + 5\log_{10}(p\,[\mathrm{mas}]) - 10
$$

視差には Gaia DR3 のゼロ点オフセット(約 −0.017 mas, Lindegren et al. 2021)を補正しておきます。

In [ ]:
PLX_ZP = -0.017  # [mas]

mw = mw_all.copy()
mw["plx"] = mw["parallax"] - PLX_ZP
mw = mw[(mw["plx"] / mw["parallax_error"] > 10) & (mw["ruwe"] < 1.4)].copy()
mw["abs_w"] = mw["w"] + 5 * np.log10(mw["plx"]) - 10
# 視差の誤差から来る絶対等級の誤差(1 次近似)
mw["abs_w_err"] = 5 / np.log(10) * mw["parallax_error"] / mw["plx"]
print(len(mw), "件")

直線 $M_W = a + b \log_{10} P$ を重み付き最小二乗法で当てはめます。
重みには、視差による誤差に加えて、周期−光度関係そのものの幅(約 0.1 mag)を入れます。
外れ値(連星や分類ミスなど)は 3σ で外して当てはめ直します。

In [ ]:
INTRINSIC_SCATTER = 0.10  # 周期-光度関係そのものの幅 [mag]


def fit_line(x, y, yerr, n_iter=5, clip=3.0):
    # 重み付き直線当てはめ + 外れ値除去。(傾き, 切片), 共分散, 使った点のマスク を返す
    sig = np.sqrt(yerr ** 2 + INTRINSIC_SCATTER ** 2)
    keep = np.ones(len(x), dtype=bool)
    for _ in range(n_iter):
        coef, cov = np.polyfit(x[keep], y[keep], 1, w=1 / sig[keep], cov="unscaled")
        resid = y - np.polyval(coef, x)
        keep = np.abs(resid) < clip * sig
    return coef, cov, keep


x = mw["logp"].to_numpy()
y = mw["abs_w"].to_numpy()
coef_mw, cov_mw, keep_mw = fit_line(x, y, mw["abs_w_err"].to_numpy())
b_mw, a_mw = coef_mw
b_err, a_err = np.sqrt(np.diag(cov_mw))
print(f"使った星: {keep_mw.sum()} / {len(mw)}")
print(f"天の川銀河: M_W = {a_mw:.3f} (±{a_err:.3f}) {b_mw:+.3f} (±{b_err:.3f}) log P")

In [ ]:
xx = np.linspace(0.2, 1.8, 100)
plt.figure(figsize=(7, 5))
plt.errorbar(x[keep_mw], y[keep_mw], yerr=mw["abs_w_err"].to_numpy()[keep_mw], fmt=".", alpha=0.5, label="Milky Way Cepheids")
plt.plot(x[~keep_mw], y[~keep_mw], "x", c="gray", label="clipped")
plt.plot(xx, np.polyval(coef_mw, xx), c="tab:red", label="fit")
plt.gca().invert_yaxis()
plt.xlabel("log10 P [day]")
plt.ylabel("M_W [mag]")
plt.legend()
plt.show()

## 3. LMC のセファイドの周期−光度関係

LMC の星はみな(ほぼ)同じ距離にあるので、見かけのヴェーゼンハイト等級をそのまま周期に対して並べるだけで、きれいな直線が見えます。
1 つ 1 つの視差は小さすぎて使えません。

In [ ]:
xl = lmc["logp"].to_numpy()
yl = lmc["w"].to_numpy()
coef_lmc, cov_lmc, keep_lmc = fit_line(xl, yl, np.zeros(len(xl)))
b_lmc, a_lmc_app = coef_lmc
print(f"使った星: {keep_lmc.sum()} / {len(lmc)}")
print(f"LMC: W = {a_lmc_app:.3f} {b_lmc:+.3f} log P")
print(f"傾きの比較: 天の川銀河 {b_mw:.3f} ± {b_err:.3f},  LMC {b_lmc:.3f} ± {np.sqrt(cov_lmc[0, 0]):.3f}")

plt.figure(figsize=(7, 5))
plt.plot(xl[keep_lmc], yl[keep_lmc], ".", ms=2, alpha=0.5, label="LMC Cepheids")
plt.plot(xl[~keep_lmc], yl[~keep_lmc], "x", c="gray", ms=3, label="clipped")
plt.plot(xx, np.polyval(coef_lmc, xx), c="tab:red", label="fit")
plt.gca().invert_yaxis()
plt.xlabel("log10 P [day]")
plt.ylabel("W [mag] (apparent)")
plt.legend()
plt.show()

LMC の星は数が多く、周期−光度関係の傾きは LMC のほうが精度よく決まります。
天の川銀河で決めた傾きと矛盾しないか見ておきましょう。

## 4. LMC までの距離

傾きを LMC で決めた値に固定し、切片(=明るさの基準点、ゼロ点)だけを天の川銀河のセファイドで決め直します。
LMC の見かけの切片と、天の川銀河で決めた絶対等級の切片の差が、LMC の距離指数です。

$$
\mu_{\mathrm{LMC}} = a_{\mathrm{LMC}}^{\mathrm{app}} - a_{\mathrm{MW}}
$$

In [ ]:
# 傾きを LMC の値に固定して、天の川銀河の切片を重み付き平均で求める
sig_mw = np.sqrt(mw["abs_w_err"].to_numpy() ** 2 + INTRINSIC_SCATTER ** 2)
k = keep_mw
wts = 1 / sig_mw[k] ** 2
a_mw_fixed = np.sum(wts * (y[k] - b_lmc * x[k])) / np.sum(wts)
a_mw_fixed_err = 1 / np.sqrt(np.sum(wts))

mu_lmc = a_lmc_app - a_mw_fixed
d_lmc = distance_from_modulus(mu_lmc)
d_lmc_err = d_lmc * np.log(10) / 5 * a_mw_fixed_err

print(f"天の川銀河の切片(傾き固定): {a_mw_fixed:.3f} ± {a_mw_fixed_err:.3f}")
print(f"LMC の距離指数: {mu_lmc:.3f} ± {a_mw_fixed_err:.3f}")
print(f"LMC までの距離: {d_lmc / 1000:.1f} ± {d_lmc_err / 1000:.1f} kpc  (約 {d_lmc * 3.26156 / 1e4:.1f} 万光年)")

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(xl[keep_lmc], yl[keep_lmc] - mu_lmc, ".", ms=2, alpha=0.4, label=f"LMC (W - {mu_lmc:.2f})")
plt.plot(x[keep_mw], y[keep_mw], "o", ms=3, c="tab:orange", label="Milky Way (from parallax)")
plt.plot(xx, a_mw_fixed + b_lmc * xx, c="tab:red", label="calibrated PL relation")
plt.gca().invert_yaxis()
plt.xlabel("log10 P [day]")
plt.ylabel("M_W [mag]")
plt.legend()
plt.show()

LMC までの距離は、食連星を使った精密な測定で $\mu = 18.477 \pm 0.026$(約 49.6 kpc, Pietrzyński et al. 2019)と求められています。

自分で出した値がこれとどれくらい合うか確認してみてください。ずれる要因としては:

- **視差のゼロ点**: セファイドのような明るい星では、Gaia の視差オフセットが −0.017 mas から少しずれることが知られている(Riess et al. 2021 など)。距離の基準点に直接効く
- **金属量**: LMC は天の川銀河より金属量が低く、周期−光度関係のゼロ点がわずかに変わる
- **ヴェーゼンハイトの係数 λ**: 減光則の仮定に依存する
- **LMC の奥行き・傾き**: LMC は天球に対して傾いていて、場所によって距離が少し違う

## このあとの距離はしご

セファイドを使えば、数千万光年先の銀河までの距離が測れます。
その銀河で **Ia 型超新星** が爆発すると、超新星の本来の明るさが較正でき、さらに遠くの銀河の距離が測れるようになります。
これが次のノートブックの内容です。

## 発展メモ

- `LAMBDA_W` を変えたり、ヴェーゼンハイト等級をやめて G 等級だけで同じことをしたりして、減光の影響の大きさを見る
- SMC(`in_smc`)でも同じように距離を出し、LMC との距離の差を見る
- `p/σ > 10` の条件をゆるめる・きつくすると、天の川銀河の較正がどう変わるか試す

## 参考・クレジット

- Gaia Collaboration, Vallenari et al. (2023), *Gaia Data Release 3: Summary of the content and survey properties*, A&A 674, A1
- Ripepi et al. (2023), *Gaia Data Release 3: Specific processing and validation of all-sky RR Lyrae and Cepheid stars – The Cepheid sample*, A&A 674, A17
- Ripepi et al. (2019), *Reclassification of Cepheids in the Gaia Data Release 2*, A&A 625, A14
- Lindegren et al. (2021), *Gaia Early Data Release 3: Parallax bias versus magnitude, colour, and position*, A&A 649, A4
- Pietrzyński et al. (2019), *A distance to the Large Magellanic Cloud that is precise to one per cent*, Nature 567, 200
- Riess et al. (2021), *Cosmic Distances Calibrated to 1% Precision with Gaia EDR3 Parallaxes and Hubble Space Telescope Photometry of 75 Milky Way Cepheids...*, ApJL 908, L6

This work has made use of data from the European Space Agency (ESA) mission *Gaia* (https://www.cosmos.esa.int/gaia), processed by the Gaia Data Processing and Analysis Consortium (DPAC, https://www.cosmos.esa.int/web/gaia/dpac/consortium).